# Multi-Tool AI Agent — Web Search + Live Weather + PDF RAG

**Mini Project:** one LangGraph agent that reads the user's question and picks the right tool.

| Tool | Built with | Used when the question is about |
|---|---|---|
| `web_search` | SerpAPI (Google) | latest news, current events, live facts from the internet |
| `get_current_weather` | OpenWeatherMap | current temperature / humidity / wind in a city |
| `search_uploaded_pdf` | FAISS + HuggingFace embeddings | anything written inside the uploaded PDF |

**Also included**

- **Last-10-message memory** — only the 10 most recent message objects are kept (tool calls and tool results count as messages).
- **LangSmith tracing** — every agent step, LLM call and tool call is recorded and viewable in the LangSmith UI.
- **Safe keys** — all keys are entered with `getpass`, never written in the code.
- **Error handling** — clear messages for a missing API key, an invalid city, or an empty result.

---

### How the agent works

```
User question
     |
     v
  Agent (LLM reads the question + system prompt)
     |
     +-- "needs live internet info?"  --> web_search           --> SerpAPI
     +-- "needs current weather?"     --> get_current_weather  --> OpenWeather API
     +-- "is it about the PDF?"       --> search_uploaded_pdf  --> FAISS vector DB
     |
     v
Tool result goes back to the LLM
     |
     v
Final natural-language answer  (whole path traced in LangSmith)
```


## 1. Install the required packages

- `langchain`, `langgraph` — agent framework and the ReAct agent graph
- `langchain-openai` — OpenAI-compatible client, pointed at OpenRouter
- `langchain-community` + `google-search-results` — SerpAPI web search
- `langchain-huggingface`, `sentence-transformers`, `faiss-cpu` — embeddings + vector database
- `pypdf` — reads the uploaded PDF
- `langsmith` — sends traces to LangSmith
- `requests` — calls the OpenWeather REST API

In [ ]:
!pip install -q -U \
    langchain langgraph langchain-openai langchain-community \
    langchain-huggingface sentence-transformers faiss-cpu \
    pypdf google-search-results langsmith requests

print("Packages installed. If Colab asks you to restart the session, restart and re-run from here.")

## 2. Add the API keys safely

Never type an API key directly into a notebook cell — anyone who sees the notebook sees the key.

We use `getpass`, which hides the text while you type. (In Colab you may also use the **key icon 🔑** in the left sidebar, *Secrets*, and read them with `google.colab.userdata`.)

You need four keys:

| Variable | Where to get it | Used for |
|---|---|---|
| `OPENROUTER_API_KEY` | https://openrouter.ai/keys | the LLM that powers the agent |
| `SERPAPI_API_KEY` | https://serpapi.com/manage-api-key | web search tool |
| `OPENWEATHER_API_KEY` | https://home.openweathermap.org/api_keys | weather tool |
| `LANGSMITH_API_KEY` | https://smith.langchain.com → Settings → API Keys | tracing |

In [ ]:
import os
import getpass

def load_key(name: str, required: bool = True) -> None:
    """Ask for one API key and store it as an environment variable."""
    value = getpass.getpass(f"Enter {name} (press Enter to skip): ").strip()
    if value:
        os.environ[name] = value
        print(f"{name} loaded")
    elif required:
        print(f"WARNING: {name} was not provided. The tool that needs it will return an error message.")
    else:
        print(f"{name} skipped")

load_key("OPENROUTER_API_KEY")
load_key("SERPAPI_API_KEY")
load_key("OPENWEATHER_API_KEY")
load_key("LANGSMITH_API_KEY")

### Turn on LangSmith tracing

Setting these environment variables is enough — LangChain and LangGraph send traces automatically.
`LANGSMITH_PROJECT` is just a folder name so your traces are easy to find.

In [ ]:
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "multi-tool-agent-mini-project"
    os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
    print("LangSmith tracing ENABLED")
    print("Project:", os.environ["LANGSMITH_PROJECT"])
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing DISABLED (no LANGSMITH_API_KEY). The agent still runs normally.")

## 3. Initialise the LLM

The LLM is the "brain" of the agent. It reads the question, decides **which tool to call**, and writes the final answer.

We use OpenRouter through the OpenAI-compatible client, exactly as in the class RAG notebook. Any tool-calling model works — `openai/gpt-4o-mini` is cheap and reliable.

In [ ]:
from langchain_openai import ChatOpenAI

MODEL_NAME = "openai/gpt-4o-mini"

if not os.environ.get("OPENROUTER_API_KEY"):
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Re-run the key cell above before continuing."
    )

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=500,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# Quick check that the key and model work
print(llm.invoke("Reply with exactly: LLM is ready").content)

## 4. Tool 1 — Web Search (SerpAPI)

The `@tool` decorator turns a normal Python function into a tool the agent can call.

**The docstring is the most important part.** The LLM never sees the function body — it reads only the name, the arguments and the docstring to decide whether this tool fits the question. So the docstring must clearly say *when to use it*.

In [ ]:
from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper


@tool
def web_search(query: str) -> str:
    """Search the live internet for current, recent or real-world information.

    Use this tool for:
    - news and recent events ("what happened...", "latest...")
    - current facts that change over time (prices, scores, who holds a position)
    - anything the model may not know and that is NOT in the uploaded PDF
      and is NOT a current-weather question.

    Do NOT use this tool for current weather (use get_current_weather)
    or for questions about the uploaded PDF (use search_uploaded_pdf).

    Args:
        query: A short, focused search query.
    """
    print(f"[TOOL CALLED] web_search -> {query}")

    api_key = os.environ.get("SERPAPI_API_KEY")
    if not api_key:
        return ("ERROR: SERPAPI_API_KEY is missing. "
                "The web search tool cannot run. Please set the key and try again.")

    try:
        search = SerpAPIWrapper(serpapi_api_key=api_key)
        result = search.run(query)
    except Exception as error:
        return f"ERROR: The web search failed ({type(error).__name__}: {error})."

    if not result or not str(result).strip():
        return "No search results were found for this query. Try rephrasing the question."

    return str(result)[:2000]   # keep the tool result small so we do not waste tokens


print("web_search tool created")

### Test the tool directly

Always test a tool on its own **before** giving it to the agent. If it fails here, the problem is the API key or the network — not the agent.

In [ ]:
print(web_search.invoke({"query": "who won the most recent ICC Cricket World Cup"}))

## 5. Tool 2 — Live Weather (OpenWeatherMap)

This tool calls the OpenWeather REST API and returns temperature, condition, humidity and wind.

HTTP status codes we handle:

| Code | Meaning | Message returned |
|---|---|---|
| 200 | Success | the weather report |
| 401 | Invalid / unauthorised key | "API key is invalid" |
| 404 | City not found | "Could not find a city named ..." |
| 429 | Too many requests | "rate limit reached" |

In [ ]:
import requests


@tool
def get_current_weather(city: str) -> str:
    """Get the CURRENT live weather for one city.

    Use this tool whenever the user asks about temperature, humidity, wind,
    rain, or general weather conditions "right now" / "today" in a place.

    Do NOT use this tool for weather history or forecasts of future days,
    and do NOT use it for general web questions.

    Args:
        city: City name, optionally with a country code, e.g. "Jammu" or "Paris,FR".
    """
    print(f"[TOOL CALLED] get_current_weather -> {city}")

    api_key = os.environ.get("OPENWEATHER_API_KEY")
    if not api_key:
        return ("ERROR: OPENWEATHER_API_KEY is missing. "
                "The weather tool cannot run. Please set the key and try again.")

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": city, "appid": api_key, "units": "metric"}

    try:
        response = requests.get(url, params=params, timeout=20)
    except requests.RequestException as error:
        return f"ERROR: Could not reach the weather service ({error})."

    if response.status_code == 401:
        return "ERROR: The OpenWeather API key is invalid or not yet activated."
    if response.status_code == 404:
        return (f"ERROR: Could not find a city named '{city}'. "
                "Please check the spelling or add a country code, e.g. 'Jammu,IN'.")
    if response.status_code == 429:
        return "ERROR: OpenWeather rate limit reached. Please wait a moment and try again."
    if response.status_code != 200:
        return f"ERROR: Weather API returned status {response.status_code}."

    data = response.json()
    name = data.get("name", city)
    country = data.get("sys", {}).get("country", "")
    condition = data.get("weather", [{}])[0].get("description", "unknown")
    main = data.get("main", {})

    return (
        f"Current weather in {name}, {country}:\n"
        f"- Condition: {condition}\n"
        f"- Temperature: {main.get('temp', 'unknown')} C\n"
        f"- Feels like: {main.get('feels_like', 'unknown')} C\n"
        f"- Humidity: {main.get('humidity', 'unknown')}%\n"
        f"- Wind speed: {data.get('wind', {}).get('speed', 'unknown')} m/s"
    )


print("get_current_weather tool created")

### Test the weather tool — a valid city and an invalid city

In [ ]:
print(get_current_weather.invoke({"city": "Jammu"}))
print()
print(get_current_weather.invoke({"city": "Xyzabc123"}))   # invalid city -> friendly error

## 6. Tool 3 — PDF RAG (FAISS vector database)

**RAG = Retrieval Augmented Generation.** The LLM does not know your PDF, so we:

1. **Load** the PDF (one `Document` per page)
2. **Split** it into small overlapping chunks
3. **Embed** each chunk into a vector (a list of numbers that captures meaning)
4. **Store** the vectors in **FAISS**
5. **Retrieve** the chunks closest in meaning to the question, and let the LLM answer from them

### 6.1 Upload the PDF

In [ ]:
from langchain_core.documents import Document

pdf_path = None

try:
    from google.colab import files
    print("Choose a PDF file to upload (or press Cancel to use the built-in sample text).")
    uploaded = files.upload()
    if uploaded:
        pdf_path = list(uploaded.keys())[0]
        print("Uploaded:", pdf_path)
except Exception as error:
    print("Colab upload not available:", error)

if not pdf_path:
    print("No PDF uploaded - the notebook will use a small built-in sample document instead.")

### 6.2 Load and split the document

`RecursiveCharacterTextSplitter` splits on paragraphs first, then lines, then spaces — so chunks stay readable.

- `chunk_size=500` — about 500 characters per chunk
- `chunk_overlap=50` — chunks share 50 characters so a sentence is not cut in half

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

SAMPLE_TEXT = """
IIT Jammu Mini Project - Multi Tool Agent.
The project builds a single AI agent with three tools: a web search tool using SerpAPI,
a live weather tool using the OpenWeatherMap API, and a PDF question-answering tool
built with FAISS. The agent keeps only the latest ten chat messages so that the token
cost stays low, and every run is traced in LangSmith.
The project was submitted for the Generative AI course.
"""

if pdf_path:
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f"Loaded {len(docs)} pages from {pdf_path}")
else:
    docs = [Document(page_content=SAMPLE_TEXT, metadata={"source": "sample"})]
    print("Using the built-in sample document")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
print("\nFirst chunk preview:\n", chunks[0].page_content[:300])

### 6.3 Create embeddings and store them in FAISS

`all-MiniLM-L6-v2` is a small, free, local embedding model — no API key needed. Each chunk becomes a 384-number vector.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})   # return the 3 closest chunks

print("Vectors stored in FAISS:", vectorstore.index.ntotal)
print("Vector dimension:", len(vectorstore.index.reconstruct(0)))

### 6.4 Wrap the retriever in a tool

The tool returns the retrieved text to the LLM, which then writes the final answer in plain language.

In [ ]:
@tool
def search_uploaded_pdf(query: str) -> str:
    """Answer questions using the content of the PDF document the user uploaded.

    Use this tool whenever the user asks about "the PDF", "the document",
    "the report", "the paper", "this file", or about any topic that would be
    written inside the uploaded document.

    Do NOT use this tool for live weather or for current internet news.

    Args:
        query: The user's question, in their own words.
    """
    print(f"[TOOL CALLED] search_uploaded_pdf -> {query}")

    try:
        results = vectorstore.similarity_search_with_relevance_scores(query, k=3)
    except Exception as error:
        return f"ERROR: Could not search the document ({error})."

    if not results:
        return "The document does not contain anything related to this question."

    # Drop very weak matches so the LLM is not misled by irrelevant text
    good = [(doc, score) for doc, score in results if score > 0.15]
    if not good:
        return ("No relevant content was found in the uploaded PDF for this question. "
                "The answer is probably not in the document.")

    parts = []
    for number, (doc, score) in enumerate(good, start=1):
        page = doc.metadata.get("page", "n/a")
        parts.append(f"[Extract {number} | page {page} | score {score:.2f}]\n{doc.page_content}")

    return "\n\n".join(parts)


print("search_uploaded_pdf tool created")

In [ ]:
print(search_uploaded_pdf.invoke({"query": "What is this document about?"}))

## 7. The system prompt — teaching the agent when to use each tool

The system prompt is the agent's rulebook. Good tool descriptions plus a clear system prompt are what make the agent pick the **right** tool.

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant with exactly three tools.

TOOL SELECTION RULES

1. get_current_weather
   Use it ONLY for current weather in a city: temperature, humidity, wind,
   rain, "how hot is it", "is it raining".
   If the user says "there" or "that city", take the city from the earlier
   conversation.
   If no city is known at all, ask the user for the city instead of guessing.

2. search_uploaded_pdf
   Use it for anything about the uploaded PDF / document / report / file,
   and for questions whose answer would plausibly be written inside it.
   If the tool says nothing relevant was found, tell the user honestly that
   the document does not cover it. Do not invent content.

3. web_search
   Use it for live or recent information from the internet: news, current
   events, latest results, prices, who currently holds a position.
   Use it when the answer is not in the PDF and is not a weather question.

GENERAL RULES
- Choose only ONE tool per step, and only if a tool is actually needed.
- If you can answer from ordinary knowledge or from the earlier conversation
  (for example "what did I ask first?"), answer directly with NO tool.
- Never invent weather values, news or document content.
- If a tool returns a message starting with ERROR, explain the problem to the
  user in one simple sentence. Do not retry endlessly.
- Keep answers short, clear and beginner-friendly.
"""

print(SYSTEM_PROMPT)

## 8. Build the agent

`create_react_agent` builds a small LangGraph with two nodes:

```
        +-------+        tool call needed        +-------+
START -->| agent | ------------------------------>| tools |
        +-------+                                +-------+
            ^                                        |
            +----------------------------------------+
            |
        no tool needed
            |
            v
           END
```

The loop repeats until the LLM answers without asking for a tool.

In [ ]:
from langgraph.prebuilt import create_react_agent

TOOLS = [web_search, get_current_weather, search_uploaded_pdf]

try:
    # Current LangGraph versions
    agent = create_react_agent(model=llm, tools=TOOLS, prompt=SYSTEM_PROMPT)
except TypeError:
    # Older LangGraph versions used state_modifier
    agent = create_react_agent(model=llm, tools=TOOLS, state_modifier=SYSTEM_PROMPT)

print("Agent created with tools:", [t.name for t in TOOLS])

## 9. Memory — keep only the latest 10 messages

One question can produce several message objects:

```
HumanMessage        "What is the weather in Jammu?"
AIMessage           (tool call: get_current_weather)
ToolMessage         (the weather report)
AIMessage           "It is 31 C and humid."
```

That is **4 messages for one question**. If we resend everything, the input grows every turn:

```
More history -> more input tokens -> higher cost -> slower replies
```

So after every turn we keep only the **last 10 message objects**.

`trim_messages` does this safely: `start_on="human"` makes sure the trimmed list never begins with a dangling tool result, which would make the API reject the request.

In [ ]:
from langchain_core.messages import HumanMessage, trim_messages

MAX_MESSAGES = 10
chat_history = []


def trim_to_last_10(messages):
    """Keep only the latest 10 message objects, without breaking tool-call pairs."""
    return trim_messages(
        messages,
        strategy="last",          # keep the newest messages
        token_counter=len,        # len => count MESSAGES, not tokens
        max_tokens=MAX_MESSAGES,  # the limit: 10 messages
        start_on="human",         # never start with an orphan tool result
        include_system=False,     # the system prompt is added by the agent itself
        allow_partial=False,
    )


def ask_agent(question: str, show_history_size: bool = True) -> str:
    """Send a question to the agent, then trim the stored history to 10 messages."""
    global chat_history

    messages = chat_history + [HumanMessage(content=question)]
    result = agent.invoke({"messages": messages})

    before = len(result["messages"])
    chat_history = trim_to_last_10(result["messages"])

    answer = chat_history[-1].content if chat_history else result["messages"][-1].content

    print(f"\nUSER: {question}")
    print(f"AGENT: {answer}")
    if show_history_size:
        print(f"(messages after this turn: {before} -> kept in memory: {len(chat_history)})")
    return answer


def clear_history() -> None:
    global chat_history
    chat_history = []
    print("Chat history cleared")


print("Memory helpers ready. Limit =", MAX_MESSAGES, "messages")

# 10. Test questions

Each test below is designed to force the agent to pick a **different** tool.

In [ ]:
clear_history()

### Test 1 — Web search question
Expected tool: **web_search**

In [ ]:
ask_agent("What are the latest news headlines about artificial intelligence in India?")

### Test 2 — Weather question
Expected tool: **get_current_weather**

In [ ]:
ask_agent("What is the current weather in Jammu?")

### Test 3 — Follow-up that depends on memory
"there" only makes sense if the previous messages are still in memory.
Expected tool: **get_current_weather** (or a direct answer from the earlier tool result).

In [ ]:
ask_agent("What is the humidity there?")

### Test 4 — PDF question
Expected tool: **search_uploaded_pdf**

In [ ]:
ask_agent("According to the uploaded PDF, what is the document about?")

### Test 5 — A second PDF question
Ask something you know is written in **your** PDF. Edit the question below to match your file.

In [ ]:
ask_agent("Summarise the main points of the PDF in three bullet points.")

### Test 6 — No tool needed
The agent should answer directly, without calling any tool.

In [ ]:
ask_agent("Thanks! In one line, what are the three tools you can use?")

### Test 7 — Error handling: invalid city
The weather tool returns a friendly error and the agent explains it instead of crashing.

In [ ]:
ask_agent("What is the weather in Zzzqqxx city?")

## 11. Proof that only the latest 10 messages are stored

Tool calls and tool results are counted as messages too.

In [ ]:
print("Messages currently stored:", len(chat_history), "(limit =", MAX_MESSAGES, ")\n")

for number, message in enumerate(chat_history, start=1):
    kind = message.__class__.__name__
    content = (message.content or "").replace("\n", " ")
    if len(content) > 90:
        content = content[:90] + "..."

    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        called = ", ".join(tc["name"] for tc in tool_calls)
        content = f"(requests tool: {called})"

    print(f"{number:2}. {kind:15} | {content}")

assert len(chat_history) <= MAX_MESSAGES, "Memory limit was exceeded!"
print("\nMemory limit respected: at most 10 message objects are kept.")

## 12. LangSmith tracing — proof that the agent and tools are traced

Every `agent.invoke()` above created a trace. Open it here:

1. Go to **https://smith.langchain.com**
2. Open **Tracing Projects**
3. Select the project **`multi-tool-agent-mini-project`**
4. Open any run and expand the tree — you will see:
   - the input messages and the system prompt
   - the LLM call and the **tool it chose**
   - the tool input (e.g. `{"city": "Jammu"}`) and the tool output
   - the final answer, latency and token usage

The cell below waits for the traces to finish uploading and then prints the most recent runs
directly from the LangSmith API — that is programmatic proof that tracing is working.

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

wait_for_all_tracers()
print("All traces submitted to LangSmith\n")

if os.environ.get("LANGSMITH_TRACING") == "true":
    from langsmith import Client

    client = Client()
    project = os.environ["LANGSMITH_PROJECT"]

    try:
        runs = list(client.list_runs(project_name=project, limit=15))
        print(f"Recent runs in project '{project}':\n")
        for run in runs:
            print(f"- {run.run_type:9} | {run.name:22} | status={run.status}")
        print(f"\nOpen the project UI: https://smith.langchain.com/  ->  {project}")
    except Exception as error:
        print("Could not read runs from LangSmith:", error)
        print("Open https://smith.langchain.com manually to view the traces.")
else:
    print("Tracing is disabled because LANGSMITH_API_KEY was not provided.")

## 13. Optional — interactive chat

Type `exit` to stop, or `clear` to erase the stored history.

In [ ]:
while True:
    question = input("You: ").strip()

    if question.lower() in {"exit", "quit"}:
        wait_for_all_tracers()
        print("Chat ended.")
        break

    if question.lower() == "clear":
        clear_history()
        continue

    if question:
        ask_agent(question)

## Summary

| Requirement | How it is met in this notebook |
|---|---|
| Search API | `web_search` tool using **SerpAPI** (section 4) |
| Weather API | `get_current_weather` tool using **OpenWeatherMap** (section 5) |
| PDF RAG | Upload -> split -> HuggingFace embeddings -> **FAISS** -> retriever tool (section 6) |
| One agent, three tools | `create_react_agent` with all three tools (section 8) |
| Correct tool selection | Detailed tool docstrings + system prompt rules (sections 4-7) |
| Only the latest 10 messages | `trim_to_last_10()` after every turn, verified in section 11 |
| LangSmith tracing | Env vars in section 2, runs listed from the API in section 12 |
| No hard-coded keys | `getpass` for all four keys (section 2) |
| Error handling | Missing key, invalid city, HTTP 401/404/429, empty search, no relevant PDF chunk |

**Key ideas to remember**

- The LLM chooses a tool by reading its **name + arguments + docstring**, so descriptions are part of the code that matters.
- The model has no memory of its own; our Python list `chat_history` is the memory, and we resend it each turn.
- Keeping only 10 messages trades a little context for much lower token cost and faster replies.
- LangSmith does not change the answer — it is a CCTV camera on the agent's decisions.
